# Customer login → Redis session ownership → AgentCore

Run all cells in order, then open http://localhost:8000. Keep this local demo on loopback.
Cognito authenticates the customer; the backend's AWS profile authorizes AgentCore calls.
Tokens and credentials remain server-side. No login secrets are printed.

**Current runtime boundary:** the local backend verifies customer ownership before invocation.
The deployed agent still keys orders by chat session ID, not Cognito customer ID.
This notebook does not change that runtime contract or migrate orders between sessions.


In [1]:
%pip install -q boto3 redis authlib itsdangerous fastapi uvicorn httpx python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import secrets
from pathlib import Path
from uuid import uuid4

import boto3
import uvicorn
from authlib.integrations.starlette_client import OAuth, OAuthError
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import HTMLResponse, RedirectResponse, JSONResponse
from pydantic import BaseModel, Field, ConfigDict
from redis import Redis
from redis.exceptions import RedisError
from starlette.middleware.sessions import SessionMiddleware

load_dotenv(Path.cwd() / ".env")
required = ["COGNITO_REGION", "COGNITO_USER_POOL_ID", "COGNITO_CLIENT_ID",
            "COGNITO_CLIENT_SECRET", "COGNITO_REDIRECT_URI", "REDIS_URL"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise ValueError("Missing configuration: " + ", ".join(missing))

REGION = os.environ["COGNITO_REGION"]
POOL = os.environ["COGNITO_USER_POOL_ID"]
ISSUER = f"https://cognito-idp.{REGION}.amazonaws.com/{POOL}"
CALLBACK = os.environ["COGNITO_REDIRECT_URI"]
assert CALLBACK == "http://localhost:8000/auth/callback", "Use the registered local callback."
redis = Redis.from_url(os.environ["REDIS_URL"], decode_responses=True,
                       socket_connect_timeout=5, socket_timeout=5)
LOGIN_COOKIE = "kiosk_demo_login"
print("Configuration loaded. Secret values are not displayed.")


Configuration loaded. Secret values are not displayed.


## 1. Frontend: customer login and chat

This HTML is served by the notebook's local backend. The browser gets an HttpOnly
login cookie and a separate chat session ID. It never receives AWS keys or Cognito tokens.
A same-origin JSON request header protects the state-changing demo routes from cross-site forms.


In [3]:
LOGIN_PAGE = """
<!doctype html><html><head><meta charset="utf-8"><title>Kiosk login demo</title></head>
<body style="font:18px system-ui;max-width:750px;margin:50px auto">
<h1>Kiosk customer login</h1>
<p><a href="/login">Sign in / Sign up with Cognito</a></p>
<button onclick="newChat()">Start new chat</button>
<p id="identity">Sign in, then start a chat.</p>
<input id="prompt" placeholder="Your message" style="width:70%">
<button onclick="send()">Send</button>
<pre id="result" style="white-space:pre-wrap"></pre>
<form method="post" action="/logout"><button>Local sign out</button></form>
<script>
let sessionId = null;
async function call(path, body) {
  const r = await fetch(path, {method:'POST',
    headers:{'Content-Type':'application/json','X-Kiosk-Request':'chat'},
    body:JSON.stringify(body)});
  const data = await r.json();
  if (!r.ok) throw new Error(data.detail || 'Request failed');
  return data;
}
async function newChat() {
  try {
    const data = await call('/sessions', {});
    sessionId = data.session_id;
    document.getElementById('identity').textContent =
      'Customer: ' + data.customer_id + ' | Chat: ' + sessionId;
    document.getElementById('result').textContent = 'Chat created. Redis ownership saved.';
  } catch(e) {document.getElementById('result').textContent = e.message;}
}
async function send() {
  try {
    if (!sessionId) throw new Error('Start a chat first.');
    const data = await call('/chat', {session_id:sessionId,
      prompt:document.getElementById('prompt').value});
    document.getElementById('result').textContent = JSON.stringify(data,null,2);
  } catch(e) {document.getElementById('result').textContent = e.message;}
}
</script></body></html>
"""


## 2. Backend: Cognito verification and Redis ownership

Authlib performs the authorization-code exchange and validates the ID token using
Cognito metadata/signing keys, audience, issuer, expiry and the login nonce/state.
The customer ID is the verified `sub` from this single configured user pool.

Redis keys:
- `kiosk:demo:login:<random cookie>` → authenticated customer, expires with the ID token.
- `kiosk:demo:owner:<session_id>` → customer ID. This demo retains ownership so an old chat cannot be reassigned after history expires.

An existing chat is authorized only if its saved owner matches the current login.
This is ownership validation, not lookup of an existing customer profile in SQLite.


In [4]:
app = FastAPI(title="Kiosk Cognito demo")
# This signed cookie contains only temporary OAuth state/nonce, never login tokens.
app.add_middleware(SessionMiddleware, secret_key=secrets.token_urlsafe(32),
                   session_cookie="kiosk_oauth_state", same_site="lax",
                   https_only=False)  # HTTP localhost only; use HTTPS cookies when deployed.
oauth = OAuth()
oauth.register(name="cognito", client_id=os.environ["COGNITO_CLIENT_ID"],
    client_secret=os.environ["COGNITO_CLIENT_SECRET"],
    server_metadata_url=ISSUER + "/.well-known/openid-configuration",
    client_kwargs={"scope": "openid email", "code_challenge_method": "S256"})

@app.get("/", response_class=HTMLResponse)
def home():
    return LOGIN_PAGE

@app.get("/login")
async def login(request: Request):
    return await oauth.cognito.authorize_redirect(request, CALLBACK)

@app.get("/auth/callback")
async def callback(request: Request):
    try:
        token = await oauth.cognito.authorize_access_token(request)
        identity = token.get("userinfo")
        if not identity or not identity.get("sub") or identity.get("token_use") != "id":
            raise HTTPException(401, "Expected a verified Cognito ID token.")
        import time
        ttl = int(identity["exp"] - time.time())
        if ttl <= 0:
            raise HTTPException(401, "Login expired. Sign in again.")
        login_id = secrets.token_urlsafe(32)
        # Keep only verified identity; this demo discards tokens after verification.
        redis.set("kiosk:demo:login:" + login_id,
                  json.dumps({"customer_id": identity["sub"]}), ex=ttl)
        old_login = request.cookies.get(LOGIN_COOKIE)
        if old_login:
            redis.delete("kiosk:demo:login:" + old_login)
        request.session.clear()
        response = RedirectResponse("/", status_code=303)
        response.set_cookie(LOGIN_COOKIE, login_id, httponly=True,
                            samesite="lax", secure=False, max_age=ttl)
        return response
    except OAuthError:
        raise HTTPException(401, "Login verification failed. Start login again.") from None


def get_customer_id(request):
    login_id = request.cookies.get(LOGIN_COOKIE)
    saved = redis.get("kiosk:demo:login:" + login_id) if login_id else None
    if not saved:
        raise HTTPException(401, "Sign in first or renew your expired login.")
    return json.loads(saved)["customer_id"]


def check_browser_request(request):
    if request.headers.get("X-Kiosk-Request") != "chat":
        raise HTTPException(403, "Use the same-origin chat page.")
    if request.headers.get("origin") not in (None, "http://localhost:8000"):
        raise HTTPException(403, "Unexpected request origin.")


def check_session_owner(session_id, customer_id):
    owner = redis.get("kiosk:demo:owner:" + session_id)
    if owner != customer_id:
        raise HTTPException(403, "This chat does not belong to your login.")


@app.post("/sessions")
def new_session(request: Request):
    check_browser_request(request)
    customer_id = get_customer_id(request)
    session_id = str(uuid4())
    if not redis.set("kiosk:demo:owner:" + session_id, customer_id, nx=True):
        raise HTTPException(409, "Retry creating a conversation.")
    return {"session_id": session_id, "customer_id": customer_id}


class ChatInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    prompt: str = Field(min_length=1, max_length=4000)
    session_id: str = Field(min_length=33, max_length=256)


@app.post("/chat")
def chat(request: Request, body: ChatInput):
    check_browser_request(request)
    customer_id = get_customer_id(request)
    check_session_owner(body.session_id, customer_id)
    # customer_id is verified locally. Current runtime accepts prompt only.
    return invoke_agent(body.prompt, body.session_id)


@app.post("/logout")
def logout(request: Request):
    if request.headers.get("origin") not in (None, "http://localhost:8000"):
        raise HTTPException(403, "Unexpected request origin.")
    login_id = request.cookies.get(LOGIN_COOKIE)
    if login_id:
        redis.delete("kiosk:demo:login:" + login_id)
    response = RedirectResponse("/", status_code=303)
    response.delete_cookie(LOGIN_COOKIE)
    return response


@app.exception_handler(RedisError)
def storage_error(request, exc):
    return JSONResponse(status_code=503, content={"detail": "Login/session storage unavailable."})

@app.exception_handler(ClientError)
def aws_error(request, exc):
    return JSONResponse(status_code=502, content={"detail": "Agent invocation failed. Check AWS logs and permissions."})


## 3. Boto3: backend invokes the deployed API

The backend uses the `kiosk` AWS profile. Customers use Cognito login, not AWS keys.
The same UUID is sent as `runtimeSessionId` on every turn in a chat. AgentCore passes
it to api.py; history.py uses it to load/save conversation messages.

Only the deployed API's supported `prompt` field is sent. To bind tools to Cognito
customer IDs, we must separately extend the runtime to verify/accept trusted customer
identity. Until then, two chats owned by one customer still have separate orders.


In [5]:
aws = boto3.Session(profile_name="kiosk", region_name="eu-north-1")
agentcore = aws.client("bedrock-agentcore", config=Config(
    connect_timeout=10, read_timeout=180, retries={"max_attempts": 0}))
RUNTIME_ARN = "arn:aws:bedrock-agentcore:eu-north-1:463277661277:runtime/kiosk_class_demo-aA5WHrCAW9"


def invoke_agent(prompt, session_id):
    response = agentcore.invoke_agent_runtime(
        agentRuntimeArn=RUNTIME_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        contentType="application/json",
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    stream = response["response"]
    try:
        return json.loads(stream.read())
    finally:
        stream.close()


## Run the demo

The callback must be registered in Cognito as `http://localhost:8000/auth/callback`.
Use localhost consistently. Stop any previous listener on port 8000 before running.
Open the page, sign up/verify email or sign in, then click **Start new chat** and send
messages. Calls incur normal AWS/model charges. Local sign-out clears this app's
login; Cognito may retain its own browser SSO session.


In [6]:
import socket
import threading

if "server" in globals() and server.started and not server.should_exit:
    raise RuntimeError("Stop the existing server with the last cell first.")
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", 8000)) == 0:
        raise RuntimeError("Port 8000 is occupied. Stop its server first.")

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000,
                                      log_level="warning", access_log=False))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
print("Open http://localhost:8000 in your browser.")
# Access logs are disabled so callback authorization codes aren't printed.


Open http://localhost:8000 in your browser.


### Stop the local backend

Run this when finished. Avoid saving notebook outputs containing customer data.
Redis demo ownership records remain until intentionally removed.


In [ ]:
server.should_exit = True
server_thread.join(timeout=5)
print("Stop requested.")
